In [2]:
!pip install optuna GPopt



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.5/253.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 19.9 MB/s eta 0:00:0000:01


In [3]:
"""
Conformalized Surrogate HPO Benchmark — sklearn built-in datasets
==================================================================
Same two-stage design as the original, but uses the 4 main sklearn
classification datasets that start with `load_*`:

    load_iris          (150 samples,  4 features, 3 classes)
    load_wine          (178 samples, 13 features, 3 classes)
    load_breast_cancer (569 samples, 30 features, 2 classes)
    load_digits        (1797 samples, 64 features, 10 classes)

No downloading required — everything is bundled with scikit-learn.

Requirements:
    pip install GPopt nnetsauce optuna xgboost scikit-learn joblib autorank
"""

# ── Imports ────────────────────────────────────────────────────────────────────
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import optuna
import GPopt as gp
import nnetsauce as ns
import xgboost as xgb

from pathlib import Path
from scipy.stats import wilcoxon, friedmanchisquare
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_digits
from sklearn.utils import all_estimators
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score, make_scorer
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Configuration ──────────────────────────────────────────────────────────────
CFG = {
    # CV
    "n_folds": 5,
    "random_state": 42,

    # HPO budget (same for all methods)
    "n_init": 10,
    "n_iter": 40,          # total evaluations = n_init + n_iter

    # Stage 1 screening (cheap pass)
    # With only 4 datasets we screen on ALL of them, but with reduced budget
    "screen_n_iter": 15,
    "screen_n_folds": 3,
    "screen_top_k": 5,     # keep top-5 surrogates for Stage 2

    # XGBoost search space
    "xgb_space": {
        "n_estimators":        (50,   500),
        "max_depth":           (2,    10),
        "learning_rate":       (1e-3, 0.5),
        "subsample":           (0.5,  1.0),
        "colsample_bytree":    (0.5,  1.0),
        "min_child_weight":    (1,    10),
        "gamma":               (0.0,  5.0),
        "reg_alpha":           (0.0,  5.0),
        "reg_lambda":          (0.5,  5.0),
    },

    # Parallelism
    "n_jobs": -1,
    "results_dir": "results_sklearn",
}

PARAM_NAMES = list(CFG["xgb_space"].keys())
LOWER = np.array([v[0] for v in CFG["xgb_space"].values()], dtype=float)
UPPER = np.array([v[1] for v in CFG["xgb_space"].values()], dtype=float)


# ── Dataset loading ────────────────────────────────────────────────────────────
def load_sklearn_datasets() -> dict:
    """
    Returns a dict mapping dataset name -> (X, y) for the 4 load_* datasets.
    """
    loaders = {
        "iris":          load_iris,
        "wine":          load_wine,
        "breast_cancer": load_breast_cancer,
        "digits":        load_digits,
    }
    datasets = {}
    for name, loader in loaders.items():
        bunch = loader()
        datasets[name] = (bunch.data.astype(float), bunch.target)
        print(f"  {name:15s}: {bunch.data.shape[0]:5d} samples, "
              f"{bunch.data.shape[1]:3d} features, "
              f"{len(np.unique(bunch.target))} classes")
    return datasets


# ── Objective factory ──────────────────────────────────────────────────────────
def make_xgb_objective(X, y, n_folds: int, random_state: int):
    """param_vector -> negative balanced accuracy (to minimise)."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    scorer = make_scorer(balanced_accuracy_score)

    def objective(params: np.ndarray) -> float:
        p = dict(zip(PARAM_NAMES, params))
        p["n_estimators"]     = int(round(p["n_estimators"]))
        p["max_depth"]        = int(round(p["max_depth"]))
        p["min_child_weight"] = int(round(p["min_child_weight"]))
        model = xgb.XGBClassifier(
            **p,
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=random_state,
            verbosity=0,
        )
        try:
            scores = cross_val_score(model, X, y, cv=skf, scoring=scorer, n_jobs=1)
            return -float(np.mean(scores))
        except Exception:
            return 0.0

    return objective


# ── Surrogate catalogue ────────────────────────────────────────────────────────
def get_surrogate_catalogue() -> list[tuple[str, object]]:
    """
    All default-param sklearn regressors wrapped in ns.PredictionInterval.
    """
    catalogue = []
    for name, Cls in all_estimators(type_filter="regressor"):
        try:
            surrogate = ns.PredictionInterval(Cls())
            catalogue.append((name, surrogate))
        except Exception:
            pass
    return catalogue


# ── Method: ConformalBO ────────────────────────────────────────────────────────
SURROGATE_TIMEOUT = 120   # seconds per surrogate — kills hangers

def run_conformal_bo(objective_func, surrogate_obj, n_init, n_iter, random_state) -> dict:
    """Runs GPOpt in a subprocess so we can enforce a hard timeout."""
    import signal

    t0 = time.time()

    def _handler(signum, frame):
        raise TimeoutError("surrogate timed out")

    # Use SIGALRM on Unix for a lightweight in-process timeout
    try:
        signal.signal(signal.SIGALRM, _handler)
        signal.alarm(SURROGATE_TIMEOUT)
        try:
            opt = gp.GPOpt(
                objective_func=objective_func,
                lower_bound=LOWER,
                upper_bound=UPPER,
                acquisition="ucb",
                method="splitconformal",
                surrogate_obj=surrogate_obj,
                n_init=n_init,
                n_iter=n_iter,
                seed=random_state,
            )
            res = opt.optimize(verbose=0, ucb_tol=1e-6)
            signal.alarm(0)  # cancel alarm
            return {"best_score": -res.best_score, "best_params": res.best_params.tolist(),
                    "elapsed": time.time() - t0, "status": "ok"}
        except TimeoutError:
            return {"best_score": np.nan, "best_params": None,
                    "elapsed": time.time() - t0, "status": f"TIMEOUT>{SURROGATE_TIMEOUT}s"}
        except Exception as e:
            signal.alarm(0)
            return {"best_score": np.nan, "best_params": None,
                    "elapsed": time.time() - t0, "status": str(e)}
        finally:
            signal.alarm(0)
    except Exception as e:
        return {"best_score": np.nan, "best_params": None,
                "elapsed": time.time() - t0, "status": str(e)}


# ── Method: GP-BO ─────────────────────────────────────────────────────────────
def run_gp_bo(objective_func, n_init, n_iter, random_state) -> dict:
    t0 = time.time()
    try:
        gp_surrogate = ns.PredictionInterval(
            GaussianProcessRegressor(
                kernel=Matern(nu=2.5), alpha=1e-6,
                normalize_y=True, random_state=random_state,
            )
        )
        opt = gp.GPOpt(
            objective_func=objective_func,
            lower_bound=LOWER,
            upper_bound=UPPER,
            acquisition="ucb",
            method="splitconformal",
            surrogate_obj=gp_surrogate,
            n_init=n_init,
            n_iter=n_iter,
            seed=random_state,
        )
        res = opt.optimize(verbose=0, ucb_tol=1e-6)
        return {"best_score": -res.best_score, "best_params": res.best_params.tolist(),
                "elapsed": time.time() - t0, "status": "ok"}
    except Exception as e:
        return {"best_score": np.nan, "best_params": None,
                "elapsed": time.time() - t0, "status": str(e)}


# ── Method: Optuna TPE ────────────────────────────────────────────────────────
def run_optuna_tpe(objective_func, n_trials, random_state) -> dict:
    t0 = time.time()
    try:
        def optuna_obj(trial):
            params = np.array([
                trial.suggest_float("n_estimators",     *CFG["xgb_space"]["n_estimators"]),
                trial.suggest_float("max_depth",        *CFG["xgb_space"]["max_depth"]),
                trial.suggest_float("learning_rate",    *CFG["xgb_space"]["learning_rate"], log=True),
                trial.suggest_float("subsample",        *CFG["xgb_space"]["subsample"]),
                trial.suggest_float("colsample_bytree", *CFG["xgb_space"]["colsample_bytree"]),
                trial.suggest_float("min_child_weight", *CFG["xgb_space"]["min_child_weight"]),
                trial.suggest_float("gamma",            *CFG["xgb_space"]["gamma"]),
                trial.suggest_float("reg_alpha",        *CFG["xgb_space"]["reg_alpha"]),
                trial.suggest_float("reg_lambda",       *CFG["xgb_space"]["reg_lambda"]),
            ])
            return objective_func(params)

        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=random_state),
        )
        study.optimize(optuna_obj, n_trials=n_trials, show_progress_bar=False)
        return {"best_score": -study.best_value, "best_params": list(study.best_params.values()),
                "elapsed": time.time() - t0, "status": "ok"}
    except Exception as e:
        return {"best_score": np.nan, "best_params": None,
                "elapsed": time.time() - t0, "status": str(e)}


# ── Method: Random Search ─────────────────────────────────────────────────────
def run_random_search(objective_func, n_trials, random_state) -> dict:
    t0 = time.time()
    rng = np.random.default_rng(random_state)
    best_score, best_params = np.inf, None
    for _ in range(n_trials):
        params = rng.uniform(LOWER, UPPER)
        score = objective_func(params)
        if score < best_score:
            best_score, best_params = score, params.tolist()
    return {"best_score": -best_score, "best_params": best_params,
            "elapsed": time.time() - t0, "status": "ok"}


# ── Stage 1: Surrogate screening ──────────────────────────────────────────────
def run_stage1_screening(datasets: dict, catalogue: list, cfg: dict):
    print("\n" + "="*60)
    print("STAGE 1: Surrogate Screening (all 4 datasets, reduced budget)")
    print("="*60)

    records = []
    ds_names = list(datasets.keys())

    for ds_name in ds_names:
        X, y = datasets[ds_name]
        obj = make_xgb_objective(X, y, cfg["screen_n_folds"], cfg["random_state"])
        print(f"\n  Dataset: {ds_name}  ({len(catalogue)} surrogates)")

        for i, (sur_name, surrogate) in enumerate(catalogue, 1):
            print(f"    [{i:02d}/{len(catalogue)}] {sur_name} ...", end=" ", flush=True)
            result = run_conformal_bo(
                objective_func=obj,
                surrogate_obj=surrogate,
                n_init=cfg["n_init"],
                n_iter=cfg["screen_n_iter"],
                random_state=cfg["random_state"],
            )
            score_str = f"{result['best_score']:.4f}" if not np.isnan(result["best_score"]) else "NaN"
            print(f"acc={score_str}  t={result['elapsed']:.1f}s  [{result['status']}]")
            records.append({
                "surrogate": sur_name,
                "dataset":   ds_name,
                "balanced_acc": result["best_score"],
                "elapsed":      result["elapsed"],
                "status":       result["status"],
            })

    df = pd.DataFrame(records)
    df["rank"] = df.groupby("dataset")["balanced_acc"].rank(
        ascending=False, method="average", na_option="bottom"
    )
    summary = (
        df.groupby("surrogate")
        .agg(
            mean_balanced_acc=("balanced_acc", "mean"),
            mean_rank=("rank", "mean"),
            n_ok=("status", lambda s: (s == "ok").sum()),
        )
        .sort_values("mean_rank")
        .reset_index()
    )
    print(f"\nTop {cfg['screen_top_k']} surrogates after screening:")
    print(summary.head(cfg["screen_top_k"]).to_string(index=False))
    return summary, df


# ── Stage 2: Main comparison ──────────────────────────────────────────────────
def run_stage2_comparison(
    datasets: dict,
    top_surrogates: list[tuple[str, object]],
    cfg: dict,
) -> pd.DataFrame:
    print("\n" + "="*60)
    print("STAGE 2: Main Comparison (full budget)")
    print("="*60)

    n_total = cfg["n_init"] + cfg["n_iter"]
    records = []

    def process_dataset(ds_name):
        X, y = datasets[ds_name]
        obj = make_xgb_objective(X, y, cfg["n_folds"], cfg["random_state"])
        local = []

        for sur_name, surrogate in top_surrogates:
            result = run_conformal_bo(
                objective_func=obj,
                surrogate_obj=surrogate,
                n_init=cfg["n_init"],
                n_iter=cfg["n_iter"],
                random_state=cfg["random_state"],
            )
            local.append({"method": f"ConfBO_{sur_name}", "dataset": ds_name, **result})

        local.append({"method": "GP_BO",       "dataset": ds_name,
                      **run_gp_bo(obj, cfg["n_init"], cfg["n_iter"], cfg["random_state"])})
        local.append({"method": "Optuna_TPE",  "dataset": ds_name,
                      **run_optuna_tpe(obj, n_total, cfg["random_state"])})
        local.append({"method": "RandomSearch","dataset": ds_name,
                      **run_random_search(obj, n_total, cfg["random_state"])})
        return local

    all_records = joblib.Parallel(n_jobs=cfg["n_jobs"], verbose=5)(
        joblib.delayed(process_dataset)(ds_name) for ds_name in datasets
    )
    for batch in all_records:
        records.extend(batch)

    return pd.DataFrame(records)


# ── Statistical analysis ──────────────────────────────────────────────────────
def analyze_results(df: pd.DataFrame) -> None:
    print("\n" + "="*60)
    print("STATISTICAL ANALYSIS")
    print("="*60)

    summary = (
        df.groupby("method")["best_score"]
        .agg(["mean", "median", "std"])
        .sort_values("mean", ascending=False)
    )
    print("\nMean balanced accuracy by method:")
    print(summary.round(4).to_string())

    # Per-dataset breakdown
    print("\nBest score per dataset per method:")
    pivot = df.pivot_table(index="dataset", columns="method", values="best_score")
    print(pivot.round(4).to_string())

    # Friedman test (needs ≥3 methods and ≥3 datasets; we have 4 datasets)
    pivot_clean = pivot.dropna()
    if pivot_clean.shape[0] >= 3 and pivot_clean.shape[1] >= 3:
        stat, p = friedmanchisquare(*[pivot_clean[m].values for m in pivot_clean.columns])
        print(f"\nFriedman test: χ²={stat:.3f}, p={p:.5f}")
        print("→ " + ("Significant differences (p < 0.05)" if p < 0.05
                       else "No significant difference (p ≥ 0.05)"))
    else:
        print("\n(Friedman test skipped — not enough complete rows)")

    # Pairwise Wilcoxon: champion ConfBO vs baselines
    conf_methods = [m for m in pivot_clean.columns if m.startswith("ConfBO_")]
    if conf_methods:
        champion = pivot_clean[conf_methods].mean().idxmax()
        print(f"\nPairwise Wilcoxon signed-rank (champion = {champion}):")
        for m in pivot_clean.columns:
            if m == champion:
                continue
            try:
                stat, p = wilcoxon(pivot_clean[champion], pivot_clean[m], alternative="greater")
                sig = "✅" if p < 0.05 else "❌"
                print(f"  vs {m:35s}: W={stat:.0f}, p={p:.4f} {sig}")
            except Exception as e:
                print(f"  vs {m:35s}: error ({e})")

    # Mean ranks
    pivot_rank = pivot_clean.rank(axis=1, ascending=False, method="average")
    print("\nMean ranks across datasets (lower = better):")
    print(pivot_rank.mean().sort_values().round(3).to_string())


# ── Persistence ───────────────────────────────────────────────────────────────
def save_results(df: pd.DataFrame, tag: str, results_dir: str) -> None:
    Path(results_dir).mkdir(exist_ok=True)
    path = Path(results_dir) / f"{tag}.csv"
    df.to_csv(path, index=False)
    print(f"Saved: {path}")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    Path(CFG["results_dir"]).mkdir(exist_ok=True)

    # 1. Load datasets
    print("Loading sklearn datasets:")
    datasets = load_sklearn_datasets()
    print(f"  → {len(datasets)} datasets ready.\n")

    # 2. Build surrogate catalogue
    catalogue = get_surrogate_catalogue()
    print(f"Surrogate catalogue: {len(catalogue)} regressors.")

    # 3. Stage 1: screen all surrogates on all 4 datasets (reduced budget)
    summary_s1, raw_s1 = run_stage1_screening(datasets, catalogue, CFG)
    save_results(raw_s1,     "stage1_screening", CFG["results_dir"])
    save_results(summary_s1, "stage1_summary",   CFG["results_dir"])

    # 4. Select top-k surrogates
    top_names      = summary_s1.head(CFG["screen_top_k"])["surrogate"].tolist()
    top_surrogates = [(n, s) for n, s in catalogue if n in top_names]
    print(f"\nTop {len(top_surrogates)} surrogates for Stage 2: {top_names}")

    # 5. Stage 2: full comparison
    df_s2 = run_stage2_comparison(datasets, top_surrogates, CFG)
    save_results(df_s2, "stage2_comparison", CFG["results_dir"])

    # 6. Statistical analysis
    analyze_results(df_s2)

    print(f"\nDone. Results saved to: {CFG['results_dir']}/")


if __name__ == "__main__":
    main()

Loading sklearn datasets:
  iris           :   150 samples,   4 features, 3 classes
  wine           :   178 samples,  13 features, 3 classes
  breast_cancer  :   569 samples,  30 features, 2 classes
  digits         :  1797 samples,  64 features, 10 classes
  → 4 datasets ready.

Surrogate catalogue: 51 regressors.

STAGE 1: Surrogate Screening (all 4 datasets, reduced budget)

  Dataset: iris  (51 surrogates)
    [01/51] ARDRegression ... acc=0.9465  t=9.2s  [ok]
    [02/51] AdaBoostRegressor ... acc=0.9465  t=9.8s  [ok]
    [03/51] BaggingRegressor ... acc=0.9465  t=7.1s  [ok]
    [04/51] BayesianRidge ... acc=0.9465  t=8.4s  [ok]
    [05/51] CCA ... acc=NaN  t=3.9s  [`n_components` upper bound is 1. Got 2 instead. Reduce `n_components`.]
    [06/51] DecisionTreeRegressor ... acc=0.9465  t=6.7s  [ok]
    [07/51] DummyRegressor ... acc=0.9465  t=7.0s  [ok]
    [08/51] ElasticNet ... acc=0.9465  t=8.6s  [ok]
    [09/51] ElasticNetCV ... acc=0.9465  t=8.8s  [ok]
    [10/51] ExtraTreeRe

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[I 2026-03-25 22:49:31,764] A new study created in memory with name: no-name-18433c69-55e4-4c3b-9b42-c8f55bb8e712
[I 2026-03-25 22:49:32,011] Trial 0 finished with value: -0.9466666666666667 and parameters: {'n_estimators': 218.5430534813131, 'max_depth': 9.60571445127933, 'learning_rate': 0.09454306819536169, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 2.403950683025824, 'gamma': 0.2904180608409973, 'reg_alpha': 4.330880728874676, 'reg_lambda': 3.20501755284444}. Best is trial 0 with value: -0.9466666666666667.
[I 2026-03-25 22:49:32,420] Trial 1 finished with value: -0.9466666666666667 and parameters: {'n_estimators': 368.63266000822045, 'max_depth': 2.1646759543664196, 'learning_rate': 0.41472250004816347, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'min_child_weight': 2.636424704863906, 'gamma': 0.9170225492671691, 'reg_alpha': 1.

Saved: results_sklearn/stage2_comparison.csv

STATISTICAL ANALYSIS

Mean balanced accuracy by method:
                                    mean  median  std
method                                               
ConfBO_OrthogonalMatchingPursuitCV  0.98    0.97 0.01
ConfBO_Lars                         0.97    0.97 0.01
ConfBO_TheilSenRegressor            0.97    0.97 0.01
ConfBO_BayesianRidge                0.97    0.97 0.01
Optuna_TPE                          0.97    0.97 0.01
ConfBO_Ridge                        0.97    0.97 0.01
GP_BO                               0.97    0.97 0.01
RandomSearch                        0.97    0.96 0.01

Best score per dataset per method:
method         ConfBO_BayesianRidge  ConfBO_Lars  ConfBO_OrthogonalMatchingPursuitCV  ConfBO_Ridge  ConfBO_TheilSenRegressor  GP_BO  Optuna_TPE  RandomSearch
dataset                                                                                                                                                      
breast

[Parallel(n_jobs=-1)]: Done   4 out of   4 | elapsed: 18.9min finished


In [5]:
"""
Conformalized Surrogate HPO Benchmark — Linear Surrogates, 72 OpenML Datasets
==============================================================================
Simplified single-stage design:
  - Surrogates: linear sklearn regressors (top performers from Stage 1 screening)
  - Baselines:  GP-BO, Optuna TPE, Random Search
  - Datasets:   all 72 OpenML-CC18 reduced datasets
  - Budget:     n_init + n_iter evaluations (same for all methods)

Evaluation reports TWO metrics per method per dataset:
  1. CV balanced accuracy  — best score found during HPO (on the train split)
  2. Test balanced accuracy — final model with best params evaluated on held-out test set

The train/test split is done once per dataset (stratified, 80/20) before any
HPO runs. All methods see the same train split; the test set is only touched
once at the very end to compute the held-out score.

Requirements:
    pip install GPopt nnetsauce optuna xgboost scikit-learn joblib requests tqdm
"""

# ── Imports ────────────────────────────────────────────────────────────────────
import io
import time
import warnings
import requests
import joblib
import numpy as np
import pandas as pd
import optuna
import GPopt as gp
import nnetsauce as ns
import xgboost as xgb

from pathlib import Path
from scipy.stats import wilcoxon, friedmanchisquare
from sklearn.linear_model import (
    Ridge, RidgeCV, BayesianRidge, Lars, LinearRegression,
    TheilSenRegressor, TweedieRegressor, LassoLarsCV, LassoCV,
    ElasticNetCV, OrthogonalMatchingPursuitCV, SGDRegressor,
)
from sklearn.compose import TransformedTargetRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import BaggingRegressor
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import balanced_accuracy_score, make_scorer
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Configuration ──────────────────────────────────────────────────────────────
CFG = {
    # Data
    "data_url":   "https://github.com/thierrymoudiki/openml-cc18-reduced/raw/main/openml-cc18-Xys-2024-05-20.pkl",
    "cache_path": "openml_cc18_reduced.pkl",

    # Train/test split
    "test_size":     0.20,   # held-out test fraction
    "random_state":  42,

    # CV (on train split only)
    "n_folds": 5,

    # HPO budget (same for all methods)
    "n_init": 10,
    "n_iter": 40,            # total = n_init + n_iter = 50 evaluations

    # XGBoost search space
    "xgb_space": {
        "n_estimators":     (50,   500),
        "max_depth":        (2,    10),
        "learning_rate":    (1e-3, 0.5),
        "subsample":        (0.5,  1.0),
        "colsample_bytree": (0.5,  1.0),
        "min_child_weight": (1,    10),
        "gamma":            (0.0,  5.0),
        "reg_alpha":        (0.0,  5.0),
        "reg_lambda":       (0.5,  5.0),
    },

    # Parallelism
    "n_jobs":      -1,
    "results_dir": "results_linear",
}

PARAM_NAMES = list(CFG["xgb_space"].keys())
LOWER = np.array([v[0] for v in CFG["xgb_space"].values()], dtype=float)
UPPER = np.array([v[1] for v in CFG["xgb_space"].values()], dtype=float)

# ── Linear surrogate catalogue (top performers from Stage 1) ───────────────────
# Instantiate fresh each time they're needed (see get_surrogates()).
LINEAR_SURROGATE_NAMES = [
    "Ridge", "RidgeCV", "BayesianRidge", "Lars",
    "LinearRegression", "TransformedTargetRegressor",
    "PLSRegression", "TheilSenRegressor", "TweedieRegressor",
    "LassoLarsCV", "LassoCV", "ElasticNetCV",
    "OrthogonalMatchingPursuitCV", "SGDRegressor", "BaggingRegressor",
]

def get_surrogates() -> list[tuple[str, object]]:
    """Return fresh (name, PredictionInterval-wrapped regressor) pairs."""
    mapping = {
        "Ridge":                      Ridge(),
        "RidgeCV":                    RidgeCV(),
        "BayesianRidge":              BayesianRidge(),
        "Lars":                       Lars(),
        "LinearRegression":           LinearRegression(),
        "TransformedTargetRegressor": TransformedTargetRegressor(),
        "PLSRegression":              PLSRegression(n_components=1),
        "TheilSenRegressor":          TheilSenRegressor(),
        "TweedieRegressor":           TweedieRegressor(),
        "LassoLarsCV":                LassoLarsCV(),
        "LassoCV":                    LassoCV(),
        "ElasticNetCV":               ElasticNetCV(),
        "OrthogonalMatchingPursuitCV":OrthogonalMatchingPursuitCV(),
        "SGDRegressor":               SGDRegressor(),
        "BaggingRegressor":           BaggingRegressor(),
    }
    return [(name, ns.PredictionInterval(reg)) for name, reg in mapping.items()]


# ── Data loading ───────────────────────────────────────────────────────────────
def load_datasets(url: str, cache_path: str) -> dict:
    cache = Path(cache_path)
    if cache.exists():
        print(f"Loading datasets from cache: {cache_path}")
        return joblib.load(cache)
    print("Downloading datasets ...")
    resp = requests.Session().get(url, stream=True)
    resp.raise_for_status()
    with io.BytesIO() as buf:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            buf.write(chunk)
        buf.seek(0)
        datasets = joblib.load(buf)
    joblib.dump(datasets, cache)
    print(f"Cached to {cache_path}")
    return datasets


# ── Objective factory ──────────────────────────────────────────────────────────
def make_xgb_objective(X_train, y_train, n_folds: int, random_state: int):
    """Returns param_vector -> negative CV balanced accuracy (to minimise)."""
    skf    = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    scorer = make_scorer(balanced_accuracy_score)

    def objective(params: np.ndarray) -> float:
        p = dict(zip(PARAM_NAMES, params))
        p["n_estimators"]     = int(round(p["n_estimators"]))
        p["max_depth"]        = int(round(p["max_depth"]))
        p["min_child_weight"] = int(round(p["min_child_weight"]))
        model = xgb.XGBClassifier(
            **p,
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=random_state,
            verbosity=0,
        )
        try:
            scores = cross_val_score(
                model, X_train, y_train, cv=skf, scoring=scorer, n_jobs=1
            )
            return -float(np.mean(scores))
        except Exception:
            return 0.0

    return objective


def eval_on_test(best_params: list, X_train, y_train, X_test, y_test,
                 random_state: int) -> float:
    """Fit XGBoost with best_params on full train set, score on test set."""
    if best_params is None:
        return np.nan
    p = dict(zip(PARAM_NAMES, best_params))
    p["n_estimators"]     = int(round(p["n_estimators"]))
    p["max_depth"]        = int(round(p["max_depth"]))
    p["min_child_weight"] = int(round(p["min_child_weight"]))
    try:
        model = xgb.XGBClassifier(
            **p,
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=random_state,
            verbosity=0,
        )
        model.fit(X_train, y_train)
        return balanced_accuracy_score(y_test, model.predict(X_test))
    except Exception:
        return np.nan


# ── Methods ────────────────────────────────────────────────────────────────────
SURROGATE_TIMEOUT = 120   # seconds per surrogate call

def run_conformal_bo(objective_func, surrogate_obj, n_init, n_iter,
                     random_state) -> dict:
    import signal
    t0 = time.time()

    def _handler(signum, frame):
        raise TimeoutError()

    try:
        signal.signal(signal.SIGALRM, _handler)
        signal.alarm(SURROGATE_TIMEOUT)
        try:
            opt = gp.GPOpt(
                objective_func=objective_func,
                lower_bound=LOWER,
                upper_bound=UPPER,
                acquisition="ucb",
                method="splitconformal",
                surrogate_obj=surrogate_obj,
                n_init=n_init,
                n_iter=n_iter,
                seed=random_state,
            )
            res = opt.optimize(verbose=0, ucb_tol=1e-6)
            signal.alarm(0)
            return {
                "cv_score":    -res.best_score,
                "best_params": res.best_params.tolist(),
                "elapsed":     time.time() - t0,
                "status":      "ok",
            }
        except TimeoutError:
            return {"cv_score": np.nan, "best_params": None,
                    "elapsed": time.time() - t0,
                    "status": f"TIMEOUT>{SURROGATE_TIMEOUT}s"}
        except Exception as e:
            signal.alarm(0)
            return {"cv_score": np.nan, "best_params": None,
                    "elapsed": time.time() - t0, "status": str(e)}
        finally:
            signal.alarm(0)
    except Exception as e:
        return {"cv_score": np.nan, "best_params": None,
                "elapsed": time.time() - t0, "status": str(e)}


def run_gp_bo(objective_func, n_init, n_iter, random_state) -> dict:
    t0 = time.time()
    try:
        surrogate = ns.PredictionInterval(
            GaussianProcessRegressor(
                kernel=Matern(nu=2.5), alpha=1e-6,
                normalize_y=True, random_state=random_state,
            )
        )
        opt = gp.GPOpt(
            objective_func=objective_func,
            lower_bound=LOWER,
            upper_bound=UPPER,
            acquisition="ucb",
            method="splitconformal",
            surrogate_obj=surrogate,
            n_init=n_init,
            n_iter=n_iter,
            seed=random_state,
        )
        res = opt.optimize(verbose=0, ucb_tol=1e-6)
        return {"cv_score": -res.best_score, "best_params": res.best_params.tolist(),
                "elapsed": time.time() - t0, "status": "ok"}
    except Exception as e:
        return {"cv_score": np.nan, "best_params": None,
                "elapsed": time.time() - t0, "status": str(e)}


def run_optuna_tpe(objective_func, n_trials, random_state) -> dict:
    t0 = time.time()
    try:
        def optuna_obj(trial):
            params = np.array([
                trial.suggest_float("n_estimators",     *CFG["xgb_space"]["n_estimators"]),
                trial.suggest_float("max_depth",        *CFG["xgb_space"]["max_depth"]),
                trial.suggest_float("learning_rate",    *CFG["xgb_space"]["learning_rate"], log=True),
                trial.suggest_float("subsample",        *CFG["xgb_space"]["subsample"]),
                trial.suggest_float("colsample_bytree", *CFG["xgb_space"]["colsample_bytree"]),
                trial.suggest_float("min_child_weight", *CFG["xgb_space"]["min_child_weight"]),
                trial.suggest_float("gamma",            *CFG["xgb_space"]["gamma"]),
                trial.suggest_float("reg_alpha",        *CFG["xgb_space"]["reg_alpha"]),
                trial.suggest_float("reg_lambda",       *CFG["xgb_space"]["reg_lambda"]),
            ])
            return objective_func(params)

        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=random_state),
        )
        study.optimize(optuna_obj, n_trials=n_trials, show_progress_bar=False)
        return {"cv_score": -study.best_value,
                "best_params": list(study.best_params.values()),
                "elapsed": time.time() - t0, "status": "ok"}
    except Exception as e:
        return {"cv_score": np.nan, "best_params": None,
                "elapsed": time.time() - t0, "status": str(e)}


def run_random_search(objective_func, n_trials, random_state) -> dict:
    t0 = time.time()
    rng = np.random.default_rng(random_state)
    best_score, best_params = np.inf, None
    for _ in range(n_trials):
        params = rng.uniform(LOWER, UPPER)
        score  = objective_func(params)
        if score < best_score:
            best_score, best_params = score, params.tolist()
    return {"cv_score": -best_score, "best_params": best_params,
            "elapsed": time.time() - t0, "status": "ok"}


# ── Per-dataset worker ─────────────────────────────────────────────────────────
def process_dataset(ds_name: str, datasets: dict, cfg: dict) -> list[dict]:
    X, y = datasets[ds_name]["dataset"]
    X = np.array(X, dtype=float)

    # One stratified train/test split — same for all methods
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=cfg["test_size"],
        stratify=y,
        random_state=cfg["random_state"],
    )

    obj     = make_xgb_objective(X_train, y_train, cfg["n_folds"], cfg["random_state"])
    n_total = cfg["n_init"] + cfg["n_iter"]
    records = []

    def _record(method, result):
        test_score = eval_on_test(
            result["best_params"], X_train, y_train, X_test, y_test,
            cfg["random_state"],
        )
        records.append({
            "method":     method,
            "dataset":    ds_name,
            "cv_score":   result["cv_score"],
            "test_score": test_score,
            "elapsed":    result["elapsed"],
            "status":     result["status"],
        })

    # ── Linear conformalized surrogates ──
    for sur_name, surrogate in get_surrogates():
        result = run_conformal_bo(
            objective_func=obj,
            surrogate_obj=surrogate,
            n_init=cfg["n_init"],
            n_iter=cfg["n_iter"],
            random_state=cfg["random_state"],
        )
        _record(f"ConfBO_{sur_name}", result)

    # ── GP-BO ──
    _record("GP_BO", run_gp_bo(obj, cfg["n_init"], cfg["n_iter"], cfg["random_state"]))

    # ── Optuna TPE ──
    _record("Optuna_TPE", run_optuna_tpe(obj, n_total, cfg["random_state"]))

    # ── Random Search ──
    _record("RandomSearch", run_random_search(obj, n_total, cfg["random_state"]))

    return records


# ── Statistical analysis ───────────────────────────────────────────────────────
def analyze_results(df: pd.DataFrame) -> None:
    print("\n" + "="*60)
    print("STATISTICAL ANALYSIS")
    print("="*60)

    for metric, label in [("cv_score", "CV balanced accuracy"),
                           ("test_score", "Test balanced accuracy")]:
        print(f"\n── {label} ──")
        summary = (
            df.groupby("method")[metric]
            .agg(["mean", "median", "std"])
            .sort_values("mean", ascending=False)
        )
        print(summary.round(4).to_string())

        pivot = df.pivot_table(index="dataset", columns="method", values=metric).dropna()

        if pivot.shape[0] >= 3 and pivot.shape[1] >= 3:
            stat, p = friedmanchisquare(*[pivot[m].values for m in pivot.columns])
            print(f"\nFriedman test: χ²={stat:.3f}, p={p:.5f}")
            print("→ " + ("Significant differences (p < 0.05)" if p < 0.05
                           else "No significant difference (p ≥ 0.05)"))

        # Champion = best mean ConfBO method
        conf_cols = [m for m in pivot.columns if m.startswith("ConfBO_")]
        if conf_cols:
            champion = pivot[conf_cols].mean().idxmax()
            print(f"\nPairwise Wilcoxon vs champion ({champion}):")
            for m in sorted(pivot.columns):
                if m == champion:
                    continue
                try:
                    stat, p = wilcoxon(pivot[champion], pivot[m], alternative="greater")
                    sig = "✅" if p < 0.05 else "❌"
                    print(f"  vs {m:40s}: W={stat:6.0f}, p={p:.4f} {sig}")
                except Exception as e:
                    print(f"  vs {m:40s}: error ({e})")

        pivot_rank = pivot.rank(axis=1, ascending=False, method="average")
        print(f"\nMean ranks — {label} (lower = better):")
        print(pivot_rank.mean().sort_values().round(3).to_string())


# ── Persistence ───────────────────────────────────────────────────────────────
def save_results(df: pd.DataFrame, tag: str, results_dir: str) -> None:
    Path(results_dir).mkdir(exist_ok=True)
    path = Path(results_dir) / f"{tag}.csv"
    df.to_csv(path, index=False)
    print(f"Saved: {path}")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    Path(CFG["results_dir"]).mkdir(exist_ok=True)

    # 1. Load data
    datasets = load_datasets(CFG["data_url"], CFG["cache_path"])
    print(f"Loaded {len(datasets)} datasets.")
    print(f"Surrogates: {LINEAR_SURROGATE_NAMES}")
    print(f"Budget per method: {CFG['n_init']} init + {CFG['n_iter']} iter "
          f"= {CFG['n_init'] + CFG['n_iter']} total evaluations")
    print(f"Test fraction: {CFG['test_size']:.0%} held-out\n")

    # 2. Run all datasets in parallel
    print("="*60)
    print("MAIN COMPARISON (all 72 datasets, linear surrogates)")
    print("="*60)

    all_records = joblib.Parallel(n_jobs=CFG["n_jobs"], verbose=5)(
        joblib.delayed(process_dataset)(ds_name, datasets, CFG)
        for ds_name in datasets.keys()
    )

    records = [r for batch in all_records for r in batch]
    df = pd.DataFrame(records)
    save_results(df, "comparison", CFG["results_dir"])

    # 3. Analysis
    analyze_results(df)

    # 4. Per-dataset summary (test scores)
    print("\n── Test score pivot (dataset × method) ──")
    pivot = df.pivot_table(
        index="dataset", columns="method", values="test_score"
    ).round(3)
    print(pivot.to_string())

    print(f"\nDone. Results saved to: {CFG['results_dir']}/")


if __name__ == "__main__":
    main()

Cached to openml_cc18_reduced.pkl
Loaded 72 datasets.
Surrogates: ['Ridge', 'RidgeCV', 'BayesianRidge', 'Lars', 'LinearRegression', 'TransformedTargetRegressor', 'PLSRegression', 'TheilSenRegressor', 'TweedieRegressor', 'LassoLarsCV', 'LassoCV', 'ElasticNetCV', 'OrthogonalMatchingPursuitCV', 'SGDRegressor', 'BaggingRegressor']
Budget per method: 10 init + 40 iter = 50 total evaluations
Test fraction: 20% held-out

MAIN COMPARISON (all 72 datasets, linear surrogates)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[I 2026-03-25 23:59:41,193] A new study created in memory with name: no-name-28472918-9512-4fad-ae35-c7ff26896c51
[I 2026-03-25 23:59:41,382] Trial 0 finished with value: -0.9562388995489821 and parameters: {'n_estimators': 218.5430534813131, 'max_depth': 9.60571445127933, 'learning_rate': 0.09454306819536169, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 2.403950683025824, 'gamma': 0.2904180608409973, 'reg_alpha': 4.330880728874676, 'reg_lambda': 3.20501755284444}. Best is trial 0 with value: -0.9562388995489821.
[I 2026-03-25 23:59:41,629] Trial 1 finished with value: -0.957665659219242 and parameters: {'n_estimators': 368.63266000822045, 'max_depth': 2.1646759543664196, 'learning_rate': 0.41472250004816347, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'min_child_weight': 2.636424704863906, 'gamma': 0.9170225492671691, 'reg_alpha': 1.5

Saved: results_linear/comparison.csv

STATISTICAL ANALYSIS

── CV balanced accuracy ──
                                    mean  median  std
method                                               
ConfBO_TheilSenRegressor            0.77    0.79 0.18
ConfBO_RidgeCV                      0.76    0.80 0.18
ConfBO_LinearRegression             0.76    0.79 0.18
ConfBO_TransformedTargetRegressor   0.76    0.79 0.18
ConfBO_BayesianRidge                0.76    0.79 0.18
ConfBO_Ridge                        0.76    0.79 0.18
ConfBO_Lars                         0.76    0.79 0.18
ConfBO_LassoLarsCV                  0.76    0.79 0.18
ConfBO_ElasticNetCV                 0.76    0.79 0.18
ConfBO_LassoCV                      0.76    0.79 0.18
ConfBO_OrthogonalMatchingPursuitCV  0.76    0.79 0.18
ConfBO_TweedieRegressor             0.76    0.79 0.18
ConfBO_BaggingRegressor             0.76    0.79 0.18
Optuna_TPE                          0.75    0.78 0.18
ConfBO_PLSRegression                0.75    0.77 

[Parallel(n_jobs=-1)]: Done  72 out of  72 | elapsed: 319.0min finished
